# Laya · plan P2 (handbook chapter 09)

Kaggle settings: **GPU T4**, **Internet on**, secret **`GITHUB_TOKEN`** attached (read-only access to the repo). The branch `orchestrator/baselines-laya` must be pushed.

Order: smoke test (one question, then 5 rows) → budget check → pilot (50 validation rows) → full validation + test → report. Outputs go to `/kaggle/working/runs/laya/` (kept with *Save Version*); copy them to `orchestrator/baselines/runs/laya/` and commit.

Stop after the smoke test cells if you want the results reviewed first. Rerunning a run cell with the same id resumes it.

In [ ]:
import os, subprocess
os.environ['USE_TF'] = '0'  # chapter 09: laya.load() can hang with TensorFlow installed
RUN_ID = 'laya-multilingual-a'
OUT = '/kaggle/working/runs/laya'
DEST = '/kaggle/temp/repo'
from kaggle_secrets import UserSecretsClient
url = f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@github.com/francescalongo17set/CHAOS-SLM-transition.git"
# no check=True: its error message would print the command, token included
if subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'orchestrator/baselines-laya', url, DEST]).returncode:
    raise RuntimeError('git clone failed: is GITHUB_TOKEN attached and the branch pushed?')
subprocess.run(['git', '-C', DEST, 'log', '-1', '--format=%H %s'], check=True)

In [ ]:
%cd /kaggle/temp/repo/orchestrator/baselines
!pip install -q -r laya/requirements-laya.txt
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
print('bf16 supported:', torch.cuda.is_available() and torch.cuda.is_bf16_supported())

## 1 · Smoke test: one question on a short Italian string

Prints the whole result object (chapter 09 step 1).

In [ ]:
import json, sys
sys.path[:0] = ['laya', '../..']
import laya
from orchestrator.baselines.common.questions import QuestionConfig
from laya_adapter import to_laya
agent = laya.load('convaiinnovations/laya', subfolder='multilingual')
print('laya', laya.__version__, '| subfolder', agent.subfolder, '| revision', agent.revision)
print('max_len', agent.cfg.get('max_len'), '| head_max_len', agent.cfg.get('head_max_len'), '| dtype', agent.dtype_for(3))
q = QuestionConfig().scope_check()
result = agent.predict('Ciao, entro quanti giorni devo denunciare un sinistro con la polizza casa?', to_laya([q]))
print(json.dumps(result, indent=1, ensure_ascii=False))

## 3 · Budget check

State tokens (Laya's own tokenizer) against `max_len - head_max_len`. More than a handful over budget: stop and ask (chapter 09).

In [ ]:
from orchestrator.baselines.common import data, state
budget = agent.cfg['max_len'] - agent.cfg['head_max_len']
for split in ['validation', 'test']:
    n = [len(agent.tok(state.serialize_state(r.model_input), add_special_tokens=False)['input_ids']) for r in data.load_split(split)]
    print(f'{split}: {len(n)} rows, max {max(n)} tokens, budget {budget}, over budget {sum(x > budget for x in n)}')
del agent; import gc; gc.collect(); torch.cuda.empty_cache()  # free the GPU for the runs below

## 1b · Smoke test: 5 validation rows, one per route

In [ ]:
!python laya/run_laya.py --run-id {RUN_ID}-smoke --smoke --out {OUT}
print(open(f'{OUT}/{RUN_ID}-smoke/validation.jsonl').read())
print(open(f'{OUT}/{RUN_ID}-smoke/run.json').read())

## 4 · Pilot: 50 validation rows

In [ ]:
!python laya/run_laya.py --run-id {RUN_ID}-pilot --split validation --limit 50 --out {OUT}

## 5 · Full runs: validation and test (validated and reported by the shared script)

In [ ]:
!python laya/run_laya.py --run-id {RUN_ID} --split validation test --out {OUT}

In [ ]:
from IPython.display import Markdown
Markdown(open(f'{OUT}/{RUN_ID}/report_test.md').read())